# NumCompute Stream: streaming decision tree demo

This notebook trains a decision tree on data that arrives in chunks. It runs from top to bottom:

1. Load the Titanic CSV with `io` (including rows with missing values).
2. Split it into chunks to simulate a stream.
3. Summarise the stream with `stats.StreamingStats.update_stats`.
4. Train a preprocessing pipeline and a buffered `DecisionTreeClassifier` with `partial_fit`, through `StreamTrainer`.
5. Log and plot the metrics over time with `visualise`, and compare two `buffer_size` values.
6. Read the accumulated metrics (`metrics`) and the benchmarks.

Every chunk is **scored before the model trains on it** (test-then-train), so each accuracy is measured on passengers the model has not seen.

In [ ]:
from pathlib import Path

import numpy as np

from numcompute_stream import io, stats, visualise
from numcompute_stream.benchmarking import compare
from numcompute_stream.metrics import (
    StreamingAUC,
    StreamingConfusionMatrix,
    StreamingF1,
    StreamingPrecision,
    StreamingRecall,
)
from numcompute_stream.pipeline import ColumnSelector, FeatureUnion, Pipeline
from numcompute_stream.preprocessing import OneHotEncoder, SimpleImputer, StandardScaler
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.tree import DecisionTreeClassifier, best_split, best_split_loop

np.set_printoptions(precision=3, suppress=True)
ROOT = Path("..") if Path("../benchmark").exists() else Path(".")
DATA = ROOT / "demo" / "data" / "train.csv"

## 1. Load the CSV with `io`

`load_csv` reads number columns as floats (missing cells become NaN) and text columns with `dtype=str` (missing cells become `""`).
Loading the whole file once shows how much is missing; the stream below reads the same file chunk by chunk.

In [ ]:
print(io.read_header(DATA))

NUM_COLS = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
CAT_COLS = ["Sex", "Embarked"]
X_num = io.load_csv(DATA, usecols=NUM_COLS)
X_cat = io.load_csv(DATA, usecols=CAT_COLS, dtype=str)
y_all = io.load_csv(DATA, usecols="Survived", dtype=int)[:, 0]

print("passengers:", len(y_all), "| survival rate:", round(y_all.mean(), 3))
print("missing numbers:", dict(zip(NUM_COLS, np.isnan(X_num).sum(axis=0).tolist())))
print("blank text cells:", dict(zip(CAT_COLS, (X_cat == "").sum(axis=0).tolist())))

## 2. Split the file into chunks

`io.iter_csv_chunks` yields at most `chunksize` rows at a time and never holds the whole file.
Three readers walk the file in step (numbers, text, label), and each chunk joins them into one row array with 5 number and 2 text columns.

In [ ]:
CHUNK = 50


def titanic_stream(chunk_size=CHUNK):
    numbers = io.iter_csv_chunks(DATA, chunksize=chunk_size, usecols=NUM_COLS)
    texts = io.iter_csv_chunks(DATA, chunksize=chunk_size, usecols=CAT_COLS, dtype=str)
    labels = io.iter_csv_chunks(DATA, chunksize=chunk_size, usecols="Survived", dtype=int)
    for num, text, label in zip(numbers, texts, labels):
        yield np.hstack([num.astype(object), text.astype(object)]), label[:, 0]


chunks = list(titanic_stream())
print("chunks:", len(chunks), "| sizes:", [len(y) for _, y in chunks])
print("missing Age per chunk:", [int(np.sum(np.isnan(X[:, 1].astype(float)))) for X, _ in chunks])
print("first row of chunk 0:", chunks[0][0][0])

## 3. Streaming statistics: `update_stats`

`StreamingStats` keeps a running count, mean and variance per column (merged chunk by chunk with Chan et al.'s formula),
the running min and max, and an equal-width histogram for approximate quantiles. The histogram's range starts at the first chunk's
min and max and doubles whenever a later value falls outside it, so every value is counted and a quantile is off by at most one bin width.
NaN is ignored column by column.

In [ ]:
running = stats.StreamingStats()  # 256 bins per column
for X, _ in chunks:
    running.update_stats(X[:, [1, 4]].astype(float))  # Age and Fare

print("count:  ", running.count, "(Age has 177 missing)")
print("mean:   ", running.mean, "| whole file:", np.nanmean(X_num[:, [1, 4]], axis=0))
print("std:    ", running.std(), "| whole file:", np.nanstd(X_num[:, [1, 4]], axis=0))
print("median: ", running.median(), "(approximate) | exact:", np.nanmedian(X_num[:, [1, 4]], axis=0))
counts, edges = running.histogram()
print("bin width (= error bound):", edges[:, 1] - edges[:, 0])
print("histogram range:", edges[:, [0, -1]].tolist())

## 4. Train the pipeline incrementally with `partial_fit`

Numbers are imputed (running mean) and standardised (running mean and std). Text is imputed (running most frequent value) and one-hot encoded.
The encoder gets its categories up front, so the number of columns the tree sees never changes mid-stream.

The tree collects chunks in a buffer and **retrains from scratch on the buffer once it holds `buffer_size` samples**, then empties it.
Between retrains, predictions come from the last trained tree. Before the first retrain there is no tree, so the first chunks are not scored.

In [ ]:
def make_pipeline(buffer_size):
    numeric = Pipeline([
        ("select", ColumnSelector([0, 1, 2, 3, 4], dtype=float)),
        ("impute", SimpleImputer(strategy="mean")),
        ("scale", StandardScaler()),
    ])
    text = Pipeline([
        ("select", ColumnSelector([5, 6], dtype=str)),
        ("impute", SimpleImputer(strategy="most_frequent", missing_values="")),
        ("onehot", OneHotEncoder(categories=[["female", "male"], ["C", "Q", "S"]],
                                 handle_unknown="ignore")),
    ])
    tree = DecisionTreeClassifier(max_depth=4, buffer_size=buffer_size, random_state=0)
    return Pipeline([("features", FeatureUnion([("num", numeric), ("text", text)])),
                     ("tree", tree)])


pipe = make_pipeline(buffer_size=200)
trainer = StreamTrainer(pipe, window=3, pos_label=1)
for X_chunk, y_chunk in chunks:
    trainer.process_chunk(X_chunk, y_chunk)  # score_chunk, then fit_chunk (partial_fit)

print(pipe)
print(f"{'chunk':>5} {'acc':>6} {'cum acc':>8} {'rolling':>8} {'retrain':>8} {'buffered':>9} {'memory':>8}")
for r in trainer.log_:
    acc = "-" if r["accuracy"] is None else f"{r['accuracy']:.3f}"
    cum = "-" if r["cumulative_accuracy"] is None else f"{r['cumulative_accuracy']:.3f}"
    roll = "-" if r["rolling_accuracy"] is None else f"{r['rolling_accuracy']:.3f}"
    print(f"{r['chunk']:>5} {acc:>6} {cum:>8} {roll:>8} {r['retrained']!s:>8} "
          f"{r['n_buffered']:>9} {r['memory_bytes'] / 1024:>6.1f}KB")

In [ ]:
tree = pipe["tree"]
print("retrains:", tree.n_retrains_, "| samples in the last training:", tree.n_samples_trained_,
      "| left in the buffer:", tree.n_buffered_)
print("tree: depth", tree.depth_, "|", tree.n_nodes_, "nodes |", tree.n_leaves_, "leaves")
feature_names = ["Pclass", "Age", "SibSp", "Parch", "Fare",
                 "female", "male", "Emb C", "Emb Q", "Emb S"]
print("root split:", feature_names[tree.feature_[0]], "<=", round(tree.threshold_[0], 3))

## 5. Visualise the metrics over time (`visualise`)

The dotted lines mark the chunks where the tree retrained. Gaps at the start are chunks scored before the first tree existed.

In [ ]:
visualise.plot_metric_over_time(
    trainer.history("cumulative_accuracy"),
    title="Cumulative test-then-train accuracy (buffer_size=200)",
    ylabel="Accuracy",
    marks=trainer.history("retrained") == 1,
    save_path=ROOT / "demo" / "figures" / "cumulative_accuracy.png",
);

### Compare two buffer sizes

A smaller buffer retrains more often on less data; a larger one waits longer before it can predict at all, then trains on more data.

In [ ]:
small = StreamTrainer(make_pipeline(buffer_size=100), window=3).run(titanic_stream())
large = StreamTrainer(make_pipeline(buffer_size=400), window=3).run(titanic_stream())
visualise.compare_models(
    small.history("cumulative_accuracy"),
    large.history("cumulative_accuracy"),
    labels=("buffer_size=100", "buffer_size=400"),
    title="Cumulative accuracy: two buffer sizes",
    ylabel="Accuracy",
    save_path=ROOT / "demo" / "figures" / "compare_buffer_sizes.png",
);
for name, t in (("100", small), ("400", large)):
    print(f"buffer_size={name}: final cumulative accuracy {t.log_[-1]['cumulative_accuracy']:.3f}, "
          f"retrains {t.log_[-1]['n_retrains']}, first scored chunk "
          f"{int(np.argmax(~np.isnan(t.history('accuracy'))))}")

### Predictions against ground truth on the latest chunk

Rows are the true class (0 = died, 1 = survived), columns the predicted class.

In [ ]:
X_last, y_last = chunks[-1]
visualise.plot_predictions_vs_ground_truth(
    y_last, pipe.predict(X_last), title="Latest chunk",
    save_path=ROOT / "demo" / "figures" / "latest_chunk.png",
);

## 6. Streaming metrics and benchmarks

Each metric class has `update(y_true_chunk, y_pred_chunk)`, `result()` and `reset()`; the confusion matrix and AUC accumulate over chunks.
Here a fresh buffered model is streamed again and every chunk (after the first retrain) is scored before training.

In [ ]:
model = make_pipeline(buffer_size=200)
confusion = StreamingConfusionMatrix(labels=[0, 1])
precision, recall, f1 = StreamingPrecision(), StreamingRecall(), StreamingF1()
auc = StreamingAUC(pos_label=1)
for X_chunk, y_chunk in chunks:
    if model["tree"].n_retrains_ > 0:  # score first...
        y_pred = model.predict(X_chunk)
        for metric in (confusion, precision, recall, f1):
            metric.update(y_chunk, y_pred)
        auc.update(y_chunk, model.predict_proba(X_chunk)[:, 1])
    model.partial_fit(X_chunk, y_chunk)  # ...then train

print("confusion (rows = true 0/1):\n", confusion.result())
print(f"precision {precision.result():.3f} | recall {recall.result():.3f} | "
      f"F1 {f1.result():.3f} | AUC {auc.result():.3f} on {confusion.n_samples_} passengers")

### Loop vs vectorised split search

The tree's split search sorts every column at once and scores every cut with cumulative class counts.
The loop version runs the same algorithm one row at a time. Both must find the same split before they are timed.

In [ ]:
X_buffer = make_pipeline(200)["features"].fit_transform(np.vstack([X for X, _ in chunks[:8]]))
y_buffer = np.concatenate([y for _, y in chunks[:8]])
print("loop:      ", best_split_loop(X_buffer, y_buffer, 2))
print("vectorised:", best_split(X_buffer, y_buffer, 2))
for row in compare({"loop": lambda: best_split_loop(X_buffer, y_buffer, 2),
                    "vectorised": lambda: best_split(X_buffer, y_buffer, 2)}):
    print(f"{row['name']:>10}: {row['best'] * 1e3:.3f} ms  ({row['speedup']:.1f}x)")

The full benchmark (`python benchmark/stream_benchmarks.py`) writes `benchmark/stream_results.md`.
Its buffer-size sweep on the drifting SEA stream:

In [ ]:
results = (ROOT / "benchmark" / "stream_results.md").read_text(encoding="utf-8")
sea_table = results.split("### SEA")[1].split("###")[0]
print("SEA" + sea_table)

In [ ]:
from IPython.display import Image

Image(filename=str(ROOT / "benchmark" / "figures" / "buffer_sweep.png"), width=600)

**Reading the sweep.** Accuracy peaks at a middle buffer size. Small buffers retrain on few samples, so each tree is noisy.
Very large buffers train on more data, but wait longer before the first prediction and react slowly after the concept drifts, because each retrain only sees the last buffer.
Retraining time grows with the buffer, while the total retraining time falls because there are fewer retrains.